# 🎵 Búsqueda por Audio
### Proyecto P3 - Chatbot con PLN y Aprendizaje Automático

Este módulo permite buscar canciones similares a partir de:
- 🎙️ **Micrófono:** Graba un fragmento de audio en tiempo real
- 📁 **Archivo:** Sube un archivo de audio (.flac, .wav, .mp3, etc.)

**Cómo funciona:**
1. Se extraen características acústicas (MFCCs + Chroma) de cada canción del dataset
2. Esas características se guardan en `features_cache.pkl` (solo se calculan una vez)
3. Cuando el usuario da un audio, se extraen sus características y se comparan con el dataset
4. Se devuelven las canciones más similares por similitud del coseno

## Celda 1 — Instalación de dependencias
Ejecuta esta celda solo la primera vez

In [10]:
# Celda 1 — Instalar dependencias (ejecutar solo si hace falta)
import sys
import subprocess

paquetes = [
    "librosa",
    "sounddevice",
    "soundfile",
    "numpy",
    "pandas",
    "scikit-learn",
    "ipywidgets",
    "pyacoustid"   # opcional: para fingerprinting (requiere fpcalc/chromaprint instalado en el sistema)
]

for p in paquetes:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
    except Exception as e:
        print(f"⚠️ No se pudo instalar {p}: {e}")

print("✅ Instalación (o verificación) de dependencias completada. Nota: para fingerprinting necesitas 'fpcalc' (Chromaprint) en tu sistema.")


✅ Instalación (o verificación) de dependencias completada. Nota: para fingerprinting necesitas 'fpcalc' (Chromaprint) en tu sistema.


## Celda 2 — Imports

In [11]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import librosa
import sounddevice as sd
import soundfile as sf
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, clear_output
import ipywidgets as widgets
import io

warnings.filterwarnings('ignore')
print('✅ Imports OK')

✅ Imports OK


## Celda 3 — Clase principal `AudioSearch`

In [12]:
# Celda 2 — Clase AudioSearch mejorada (ventanas + mfcc+deltas + fingerprint opcional)
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import librosa
import sounddevice as sd
import soundfile as sf
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# Intentar importar pyacoustid (opcional)
try:
    import acoustid
    HAS_ACOUSTID = True
except Exception:
    HAS_ACOUSTID = False

warnings.filterwarnings("ignore")

class AudioSearch:
    SAMPLE_RATE = 22050
    WINDOW_SEC = 5.0        # ventana para indexado y búsqueda (segundos)
    HOP_SEC = 2.5           # solapamiento entre ventanas (segundos)
    N_MFCC = 20
    HOP_LENGTH = 512
    CACHE_FILE = "features_cache_windows.pkl"
    SCALER_FILE = "scaler_cache.pkl"
    FINGERPRINT_FILE = "fingerprint_cache.pkl"

    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.df = pd.read_csv(csv_path)
        self.song_windows = {}      # {song_idx: np.array(shape=(n_windows, n_features))}
        self.song_window_means = {} # {song_idx: np.array(mean_feature_vector)}
        self.valid_indices = []     # índices de canciones indexadas
        self.scaler = None
        self.fingerprints = {}      # {song_idx: fingerprint_string} (si está disponible)
        print(f"📀 Dataset cargado: {len(self.df)} canciones")

    # -------------------------
    # UTIL: preprocesado audio
    # -------------------------
    def _preprocess_audio_array(self, y):
        # Recortar silencios y normalizar
        y, _ = librosa.effects.trim(y, top_db=20)
        if y.size == 0:
            return y
        y = librosa.util.normalize(y)
        return y

    # -------------------------
    # EXTRACCIÓN DE FEATURES (por segmento)
    # -------------------------
    def _extract_features_from_array(self, y, sr):
        # MFCC + deltas
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.N_MFCC, hop_length=self.HOP_LENGTH)
        mfcc_delta = librosa.feature.delta(mfcc)

        # Pooling: mean, std, median para cada coeficiente
        mfcc_pool = np.concatenate([
            np.mean(mfcc, axis=1),
            np.std(mfcc, axis=1),
            np.median(mfcc, axis=1),
            np.mean(mfcc_delta, axis=1),
            np.std(mfcc_delta, axis=1)
        ])

        # Chroma pooling
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=self.HOP_LENGTH)
        chroma_pool = np.concatenate([
            np.mean(chroma, axis=1),
            np.std(chroma, axis=1),
            np.median(chroma, axis=1)
        ])

        # Centroid y ZCR
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=self.HOP_LENGTH)
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=self.HOP_LENGTH)

        centroid_mean = np.mean(centroid) if centroid.size else 0.0
        centroid_std = np.std(centroid) if centroid.size else 0.0
        zcr_mean = np.mean(zcr) if zcr.size else 0.0
        zcr_std = np.std(zcr) if zcr.size else 0.0

        feature_vector = np.concatenate([
            mfcc_pool,
            chroma_pool,
            [centroid_mean, centroid_std, zcr_mean, zcr_std]
        ])

        return feature_vector

    # -------------------------
    # Extraer ventanas de un archivo
    # -------------------------
    def _extract_windows_features(self, audio_path):
        try:
            y, sr = librosa.load(audio_path, sr=self.SAMPLE_RATE, mono=True)
            if y is None or y.size == 0:
                return np.empty((0,)), 0
            y = self._preprocess_audio_array(y)
            if y.size == 0:
                return np.empty((0,)), 0

            win = int(self.WINDOW_SEC * sr)
            hop = int(self.HOP_SEC * sr)
            feats = []
            for start in range(0, max(1, len(y) - win + 1), hop):
                seg = y[start:start + win]
                if seg.size < 256:
                    continue
                fv = self._extract_features_from_array(seg, sr)
                feats.append(fv)
            if not feats:
                # si la canción es más corta que la ventana, extraer features de todo el audio
                fv = self._extract_features_from_array(y, sr)
                return np.array([fv]), sr
            return np.array(feats), sr
        except Exception as e:
            print(f"   ⚠️ Error extrayendo ventanas: {audio_path} — {e}")
            return np.empty((0,)), 0

    # -------------------------
    # Fingerprint (opcional, requiere fpcalc/chromaprint + pyacoustid)
    # -------------------------
    def _fingerprint_file(self, path):
        if not HAS_ACOUSTID:
            return None
        try:
            # acoustid.fingerprint_file devuelve (duration, fingerprint)
            dur, fp = acoustid.fingerprint_file(path)
            return fp
        except Exception:
            return None

    # -------------------------
    # Construir índice (ventanas + scaler + fingerprints opcional)
    # -------------------------
    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(self.CACHE_FILE) and os.path.exists(self.SCALER_FILE):
            print("📦 Cargando índice y scaler desde caché...")
            with open(self.CACHE_FILE, "rb") as f:
                cache = pickle.load(f)
            with open(self.SCALER_FILE, "rb") as f:
                self.scaler = pickle.load(f)
            self.song_windows = cache["song_windows"]
            self.song_window_means = cache["song_window_means"]
            self.valid_indices = cache["valid_indices"]
            self.fingerprints = cache.get("fingerprints", {})
            print(f"✅ Índice cargado: {len(self.valid_indices)} canciones indexadas")
            return

        print(f"🔨 Construyendo índice por ventanas para {len(self.df)} canciones...")
        all_windows = []
        song_windows = {}
        song_means = {}
        valid_indices = []
        fingerprints = {}

        for idx, row in self.df.iterrows():
            audio_path = row.get("Audio_Path", "")
            print(f"   [{idx+1:02d}/{len(self.df)}] {row.get('Artista','?')} - {row.get('Titulo','?')}", end=" ... ")
            if not os.path.exists(audio_path):
                print("⚠️  archivo no encontrado")
                continue

            feats, sr = self._extract_windows_features(audio_path)
            if feats.size == 0:
                print("❌")
                continue

            song_windows[idx] = feats
            song_means[idx] = np.mean(feats, axis=0)
            valid_indices.append(idx)
            all_windows.append(feats)
            # fingerprint opcional
            fp = self._fingerprint_file(audio_path)
            if fp:
                fingerprints[idx] = fp
            print("✅")

        if not all_windows:
            print("\n❌ No se pudo indexar ninguna canción. Verifica que los archivos de audio existen.")
            return

        # Concatenar todas las ventanas para ajustar scaler
        X = np.vstack(all_windows)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Re-split scaled windows into per-song arrays
        song_windows_scaled = {}
        pos = 0
        for idx in valid_indices:
            n = song_windows[idx].shape[0]
            song_windows_scaled[idx] = X_scaled[pos:pos + n]
            pos += n

        # Guardar estructuras
        self.song_windows = song_windows_scaled
        self.song_window_means = {idx: np.mean(w, axis=0) for idx, w in song_windows_scaled.items()}
        self.valid_indices = valid_indices
        self.scaler = scaler
        self.fingerprints = fingerprints

        # Guardar caché
        with open(self.CACHE_FILE, "wb") as f:
            pickle.dump({
                "song_windows": self.song_windows,
                "song_window_means": self.song_window_means,
                "valid_indices": self.valid_indices,
                "fingerprints": self.fingerprints
            }, f)
        with open(self.SCALER_FILE, "wb") as f:
            pickle.dump(self.scaler, f)

        print(f"\n✅ Índice construido: {len(valid_indices)} canciones")
        print(f"💾 Caché guardado en {self.CACHE_FILE} y {self.SCALER_FILE}")

    # -------------------------
    # Búsqueda: fingerprint primero, luego ventanas
    # -------------------------
    def _search_by_windows(self, query_windows, top_n=5):
        # query_windows: np.array(shape=(n_q_windows, n_features)) ya sin escalar
        if self.scaler is None:
            raise RuntimeError("El índice no está construido. Ejecuta build_index() primero.")

        q_scaled = self.scaler.transform(query_windows)  # shape (n_q, n_features)

        # Para cada canción, calcular la similitud máxima entre cualquier ventana de la query y cualquier ventana de la canción
        song_scores = []
        for idx in self.valid_indices:
            windows = self.song_windows.get(idx)
            if windows is None or windows.size == 0:
                continue
            # similitud entre matrices: (n_q, n_song_windows)
            sims = cosine_similarity(q_scaled, windows)
            max_sim = np.max(sims)
            song_scores.append((idx, max_sim))

        # ordenar por score descendente
        song_scores.sort(key=lambda x: x[1], reverse=True)
        resultados = []
        for rank, (idx, score) in enumerate(song_scores[:top_n], start=1):
            row = self.df.iloc[idx]
            resultados.append({
                "Posición": rank,
                "Título": row.get("Titulo", ""),
                "Artista": row.get("Artista", ""),
                "Álbum": row.get("Album", ""),
                "Género": row.get("Genero", ""),
                "Año": row.get("Año", ""),
                "Similitud": f"{score:.4f}"
            })
        return pd.DataFrame(resultados)

    def _search(self, query_path, top_n=5):
        # 1) intentar fingerprint exacto (si disponible)
        if HAS_ACOUSTID:
            qfp = self._fingerprint_file(query_path)
            if qfp:
                # buscar coincidencia exacta (o substring) en fingerprints
                for idx, fp in self.fingerprints.items():
                    if fp and (qfp == fp or qfp in fp or fp in qfp):
                        # coincidencia exacta: devolver esa canción en primer lugar
                        row = self.df.iloc[idx]
                        return pd.DataFrame([{
                            "Posición": 1,
                            "Título": row.get("Titulo", ""),
                            "Artista": row.get("Artista", ""),
                            "Álbum": row.get("Album", ""),
                            "Género": row.get("Genero", ""),
                            "Año": row.get("Año", ""),
                            "Similitud": "1.0000 (fingerprint)"
                        }])

        # 2) si no hay fingerprint o no coincide, usar ventanas
        q_windows, sr = self._extract_windows_features(query_path)
        if q_windows.size == 0:
            return pd.DataFrame()
        return self._search_by_windows(q_windows, top_n=top_n)

    # -------------------------
    # Métodos públicos
    # -------------------------
    def search_by_file(self, audio_path: str, top_n: int = 5) -> pd.DataFrame:
        print(f"🔍 Analizando archivo: {audio_path}")
        return self._search(audio_path, top_n=top_n)

    def search_by_mic(self, duration: int = 8, top_n: int = 5) -> pd.DataFrame:
        print(f"🎙️  Grabando {duration} segundos... ¡Empieza a cantar o pon la música!")
        try:
            audio_data = sd.rec(int(duration * self.SAMPLE_RATE), samplerate=self.SAMPLE_RATE, channels=1, dtype="float32")
            sd.wait()
            audio_data = audio_data.flatten()
            temp_path = "_temp_mic_recording.wav"
            sf.write(temp_path, audio_data, self.SAMPLE_RATE)
        except Exception as e:
            print(f"❌ Error con el micrófono: {e}")
            return pd.DataFrame()

        results = self.search_by_file(temp_path, top_n=top_n)
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return results

    def search_by_bytes(self, audio_bytes: bytes, filename: str = "upload.wav", top_n: int = 5) -> pd.DataFrame:
        temp_path = f"_temp_upload{os.path.splitext(filename)[1]}"
        with open(temp_path, "wb") as f:
            f.write(audio_bytes)
        results = self.search_by_file(temp_path, top_n=top_n)
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return results

print("✅ Clase AudioSearch definida (ventanas + fingerprint opcional)")


✅ Clase AudioSearch definida (ventanas + fingerprint opcional)


## Celda 4 — Inicializar y construir el índice

> ⚠️ **Esta celda puede tardar varios minutos la primera vez** (procesa los 67 audios).
> Las siguientes veces carga el caché en segundos.

In [13]:
# Ajusta la ruta al CSV si es necesario
CSV_PATH = 'data/dataset_musica.csv'

searcher = AudioSearch(CSV_PATH)
searcher.build_index()   # Si ya existe features_cache.pkl, carga automáticamente

📀 Dataset cargado: 68 canciones
📦 Cargando índice y scaler desde caché...
✅ Índice cargado: 68 canciones indexadas


## Celda 5 — Búsqueda por archivo (modo Jupyter)

In [ ]:
# Celda 3 — Widget de búsqueda por archivo (robusto, copia/pega completa)
from IPython.display import display, clear_output
import ipywidgets as widgets

def widget_buscar_por_archivo(searcher):
    titulo = widgets.HTML("<h3>📁 Buscar por Archivo de Audio</h3>")

    uploader = widgets.FileUpload(
        accept=".wav,.mp3,.flac,.ogg,.m4a,.aac",
        multiple=False,
        description="Seleccionar audio",
        button_style="primary"
    )

    top_n_slider = widgets.IntSlider(value=5, min=1, max=10, description="Resultados:", style={"description_width": "initial"})

    btn_buscar = widgets.Button(description="🔍 Buscar canciones similares", button_style="success", layout=widgets.Layout(width="250px", height="35px"))

    output = widgets.Output()

    def on_buscar(b):
        with output:
            clear_output(wait=True)
            # Depuración: mostrar la estructura de uploader.value
            print("DEBUG uploader.value repr:", repr(uploader.value))
            if not uploader.value:
                print("⚠️  Selecciona un archivo primero.")
                return

            try:
                # Manejo robusto: el entorno puede devolver una tupla con un dict dentro
                first = list(uploader.value)[0]
                # first puede ser un dict o una tupla (meta,)
                if isinstance(first, dict) and "name" in first:
                    meta = first
                    nombre = meta.get("name")
                elif isinstance(first, tuple) and len(first) >= 1:
                    # ejemplo: ({'name':..., 'content': memoryview(...)},)
                    # en tu caso vimos: ({'name':..., 'content': <memory at ...>, ...},)
                    maybe = first[0]
                    if isinstance(maybe, dict) and "name" in maybe:
                        meta = maybe
                        nombre = meta.get("name")
                    else:
                        # fallback: intentar interpretar first as (name, meta)
                        try:
                            nombre = first[0]
                            meta = first[1]
                        except Exception:
                            print("❌ No pude interpretar uploader.value.")
                            return
                else:
                    print("❌ Estructura de uploader.value no reconocida.")
                    return

                contenido = meta.get("content") or meta.get("data")
                # convertir memoryview a bytes si hace falta
                if isinstance(contenido, memoryview):
                    contenido = contenido.tobytes()
                elif isinstance(contenido, bytearray):
                    contenido = bytes(contenido)
                elif not isinstance(contenido, (bytes,)):
                    try:
                        contenido = bytes(contenido)
                    except Exception:
                        print("❌ No pude convertir el contenido a bytes.")
                        return

            except Exception as e:
                print("❌ Error procesando uploader.value:", e)
                return

            print(f"Archivo cargado: {nombre} ({len(contenido) // 1024} KB)")
            results = searcher.search_by_bytes(contenido, filename=nombre, top_n=top_n_slider.value)
            if results.empty:
                print("❌ No se encontraron resultados.")
            else:
                print(f"\n🎵 Top {top_n_slider.value} canciones más similares:\n")
                display(results)

    btn_buscar.on_click(on_buscar)

    display(widgets.VBox([titulo, uploader, top_n_slider, btn_buscar, output]))

widget_buscar_por_archivo(searcher)


## Celda 6 — Búsqueda por micrófono (modo Jupyter)

In [ ]:
# Celda 4 — Widget de búsqueda por micrófono (copia/pega completa)
from IPython.display import display, clear_output
import ipywidgets as widgets

def widget_buscar_por_microfono(searcher):
    titulo = widgets.HTML("<h3>🎙️ Buscar por Micrófono</h3>")

    duracion_slider = widgets.IntSlider(value=8, min=3, max=15, description="Duración (s):", style={"description_width": "initial"})
    top_n_slider = widgets.IntSlider(value=5, min=1, max=10, description="Resultados:", style={"description_width": "initial"})

    btn_grabar = widgets.Button(description="⏺️  Iniciar grabación", button_style="danger", layout=widgets.Layout(width="220px", height="35px"))
    estado = widgets.HTML("")
    output = widgets.Output()

    def on_grabar(b):
        with output:
            clear_output(wait=True)
            estado.value = f'<b style="color:red">🔴 Grabando {duracion_slider.value}s... Pon la música o canta!</b>'
            btn_grabar.disabled = True

            results = searcher.search_by_mic(duration=duracion_slider.value, top_n=top_n_slider.value)

            estado.value = ""
            btn_grabar.disabled = False

            if results.empty:
                print("❌ No se encontraron resultados.")
            else:
                print(f"\n🎵 Top {top_n_slider.value} canciones más similares:\n")
                display(results)

    btn_grabar.on_click(on_grabar)

    display(widgets.VBox([titulo, duracion_slider, top_n_slider, btn_grabar, estado, output]))

widget_buscar_por_microfono(searcher)


---
## Celda 7 — Modo consola / línea de comandos
Esta celda muestra cómo usar el módulo directamente sin widgets.
Útil para integrar con los otros módulos del equipo o correr como script.

In [16]:
# ──────────────────────────────────────────────────────────
#  MODO CONSOLA — Ejecuta esto directamente para probar
# ──────────────────────────────────────────────────────────

import sys

def demo_consola():
    print('\n' + '='*50)
    print('  🎵 BÚSQUEDA POR AUDIO — Modo consola')
    print('='*50)
    print('\n¿Cómo quieres buscar?')
    print('  [1] Subir un archivo de audio')
    print('  [2] Grabar con el micrófono')
    print('  [0] Salir')

    opcion = input('\nElige una opción: ').strip()

    if opcion == '1':
        ruta = input('Ruta del archivo de audio: ').strip()
        if not os.path.exists(ruta):
            print(f'❌ Archivo no encontrado: {ruta}')
            return
        resultados = searcher.search_by_file(ruta)

    elif opcion == '2':
        try:
            seg = int(input('¿Cuántos segundos grabar? (recomendado 5-10): ').strip())
        except ValueError:
            seg = 5
        resultados = searcher.search_by_mic(duration=seg)

    elif opcion == '0':
        print('Hasta luego!')
        return
    else:
        print('Opción no válida')
        return

    if resultados.empty:
        print('\n❌ No se encontraron resultados.')
    else:
        print('\n🎵 Canciones más similares:')
        print(resultados.to_string(index=False))

# Descomentar para correr en consola:
demo_consola()


  🎵 BÚSQUEDA POR AUDIO — Modo consola

¿Cómo quieres buscar?
  [1] Subir un archivo de audio
  [2] Grabar con el micrófono
  [0] Salir
Opción no válida
